# Colombia → Australia Travel Planner
## Análisis Exploratorio de Rutas y Aeropuertos

Este notebook explora el dataset inicial de aeropuertos y rutas de vuelo entre Colombia y Australia.

> ⚠️ **Disclaimer:** La información sobre visas es orientativa. Siempre verifica con fuentes oficiales antes de comprar tus tiquetes.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.routes import load_airports, load_routes, get_routes_dataframe
from src.scoring import rank_routes, score_route
from src.map_builder import build_overview_map, build_route_map

print('Librerías cargadas correctamente ✅')

## 1. Carga de datos

In [ ]:
airports = load_airports()
routes = load_routes()

print(f'Aeropuertos cargados: {len(airports)}')
print(f'Rutas cargadas: {len(routes)}')

In [ ]:
# DataFrame de aeropuertos
ap_data = [{
    'IATA': code,
    'Ciudad': ap.city,
    'País': ap.country,
    'Lat': ap.latitude,
    'Lon': ap.longitude,
    'Turístico': ap.is_tourist_stopover,
} for code, ap in airports.items()]

df_airports = pd.DataFrame(ap_data)
df_airports.head(10)

## 2. Análisis de rutas

In [ ]:
df_routes = get_routes_dataframe(routes)
df_routes.head()

In [ ]:
# Estadísticas básicas
print('Duración promedio (horas):', df_routes['duration_hours'].mean())
print('Escalas promedio:', df_routes['total_stops'].mean())
print('\nDistribución por potencial turístico:')
print(df_routes['tourist_potential'].value_counts())
print('\nDistribución por complejidad de visa:')
print(df_routes['visa_complexity'].value_counts())

## 3. Scoring de rutas

In [ ]:
# Ranking para el mes de octubre (ideal para turismo en Asia)
ranked = rank_routes(routes, airports, departure_month=10)

print('Top 5 rutas para octubre:')
for i, (route, breakdown) in enumerate(ranked[:5], 1):
    print(f'{i}. {route.name}')
    print(f'   Score: {breakdown.total:.1f}/100 ({breakdown.label})')
    print()

In [ ]:
# Visualizar scores
scores_df = pd.DataFrame([
    {'Ruta': r.name.split('→')[-1].strip(), 'Score': b.total, 'Escalas': r.total_stops,
     'Turismo': r.tourist_potential, 'Origen': r.name.split('→')[0].strip()}
    for r, b in ranked
])

fig = px.bar(
    scores_df,
    x='Score',
    y='Ruta',
    color='Turismo',
    orientation='h',
    title='Score de rutas Colombia → Australia (Octubre)',
    color_discrete_map={'high': '#27ae60', 'medium': '#f39c12', 'low': '#e74c3c'},
    text='Score',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)
fig.show()

## 4. Mapa interactivo — todas las rutas

In [ ]:
overview_map = build_overview_map(routes, airports)
overview_map

## 5. Mapa de una ruta específica

In [ ]:
# Mejor ruta según el ranking
best_route, best_score = ranked[0]
print(f'Mejor ruta: {best_route.name} — Score: {best_score.total:.1f}')

route_map = build_route_map(best_route, airports, animated=False)
route_map

## 6. Análisis por mes de salida
¿Cómo cambia el score según el mes en que viajamos?

In [ ]:
month_names = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
               7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

monthly_scores = []
for month in range(1, 13):
    for route in routes[:5]:  # Top 5 rutas
        s = score_route(route, airports, departure_month=month)
        monthly_scores.append({
            'Mes': month_names[month],
            'Ruta': route.name.split('→')[-2].strip() if '→' in route.name else route.name[:20],
            'Score': s.total,
        })

df_monthly = pd.DataFrame(monthly_scores)

fig2 = px.line(
    df_monthly,
    x='Mes', y='Score', color='Ruta',
    title='Evolución del score por mes (top 5 rutas)',
    markers=True,
)
fig2.show()

## 7. Próximos pasos

- [ ] Integrar API de Amadeus para obtener precios reales
- [ ] Agregar más aeropuertos de escala (BKK, CGK, FRA, LHR)
- [ ] Incluir datos de frecuencias de vuelo por ruta
- [ ] Análisis de precios históricos por temporada
- [ ] Modelo de recomendación con ML basado en preferencias del usuario
- [ ] Integración con OpenFlights dataset para datos más completos